# PCFI261 - Semana 09, Clase 2
## Redes neuronales con Pima Indians Diabetes

Construiremos una red neuronal densa para un problema clasico de clasificacion binaria. La variable `Outcome` toma valores 0 o 1 y las entradas son variables clinicas tabuladas.

**Advertencia:** este notebook es material docente. No entrega diagnosticos medicos ni reemplaza una evaluacion profesional. Las predicciones para casos nuevos solo sirven para explorar como responde el modelo.

## Objetivos

1. Cargar e inspeccionar el dataset Pima Indians Diabetes.
2. Preparar datos tabulados para una red neuronal.
3. Construir una red densa con `tensorflow.keras`.
4. Comparar redes con menos y mas neuronas.
5. Interpretar matriz de confusion, AUC y umbrales.
6. Ingresar un caso nuevo de forma anonima y responsable.
7. Discutir cuando una GPU puede ser util.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
np.random.seed(261)
tf.random.set_seed(261)
print('TensorFlow:', tf.__version__)
print('GPUs disponibles:', tf.config.list_physical_devices('GPU'))

## 1. Cargar datos

El dataset tiene 768 registros, 8 variables de entrada y una salida binaria. Algunas columnas tienen ceros que no son fisiologicamente razonables; mas adelante los trataremos como datos faltantes.

In [ ]:
url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)
df.head()

In [ ]:
df.info()
display(df.describe().T)
display(df['Outcome'].value_counts(normalize=True).rename('fraccion'))

### Actividad 1

Identifica las variables de entrada, la variable objetivo y las escalas de cada columna. Discute por que `Glucose = 0` o `BMI = 0` no deberian tratarse como mediciones normales.

## 2. Preprocesamiento

Separaremos train, validation y test. Luego ajustaremos imputacion por mediana y escalamiento usando solo train. Esto evita fuga de informacion desde validation o test hacia el entrenamiento.

In [ ]:
feature_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']
zero_as_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df_clean = df.copy()
df_clean[zero_as_missing] = df_clean[zero_as_missing].replace(0, np.nan)
X = df_clean[feature_names].copy()
y = df_clean['Outcome'].astype(int).copy()
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=261, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=261, stratify=y_temp)
preprocess = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
X_train_s = preprocess.fit_transform(X_train)
X_val_s = preprocess.transform(X_val)
X_test_s = preprocess.transform(X_test)
X_train_s.shape, X_val_s.shape, X_test_s.shape

## 3. Primera red neuronal

Una capa densa calcula una transformacion lineal seguida de una activacion no lineal. Para clasificacion binaria usamos una salida sigmoide y `binary_crossentropy`.

In [ ]:
def build_mlp(hidden_layers=(16, 8), learning_rate=1e-3, dropout=0.0):
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_s.shape[1],)))
    for neurons in hidden_layers:
        model.add(layers.Dense(neurons, activation='relu'))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss='binary_crossentropy', metrics=['accuracy', keras.metrics.AUC(name='auc')])
    return model

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
model = build_mlp((16, 8), learning_rate=1e-3)
model.summary()
history = model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=200, batch_size=32, callbacks=[early_stop], verbose=0)
pd.DataFrame(history.history).tail()

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist['loss'], label='train'); axes[0].plot(hist['val_loss'], label='validation'); axes[0].set_title('loss'); axes[0].legend()
axes[1].plot(hist['accuracy'], label='train'); axes[1].plot(hist['val_accuracy'], label='validation'); axes[1].set_title('accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Comparar arquitecturas

Entrenaremos una red muy pequena, una base, una con mas neuronas y otra regularizada. La pregunta no es cual gana por azar, sino como cambia el comportamiento al modificar la capacidad del modelo.

In [ ]:
experiments = [
    {'name': 'muy pequena', 'hidden_layers': (4,), 'learning_rate': 1e-3, 'dropout': 0.0},
    {'name': 'base', 'hidden_layers': (16, 8), 'learning_rate': 1e-3, 'dropout': 0.0},
    {'name': 'mas neuronas', 'hidden_layers': (64, 32), 'learning_rate': 1e-3, 'dropout': 0.0},
    {'name': 'regularizada', 'hidden_layers': (64, 32), 'learning_rate': 1e-3, 'dropout': 0.25},
]
results = []
trained_models = {}
for cfg in experiments:
    tf.keras.backend.clear_session()
    tf.random.set_seed(261)
    exp_model = build_mlp(cfg['hidden_layers'], cfg['learning_rate'], cfg['dropout'])
    exp_history = exp_model.fit(X_train_s, y_train, validation_data=(X_val_s, y_val), epochs=200, batch_size=32, callbacks=[early_stop], verbose=0)
    val_loss, val_acc, val_auc = exp_model.evaluate(X_val_s, y_val, verbose=0)
    trained_models[cfg['name']] = exp_model
    results.append({'modelo': cfg['name'], 'capas': cfg['hidden_layers'], 'dropout': cfg['dropout'], 'parametros': exp_model.count_params(), 'epochs': len(exp_history.history['loss']), 'val_loss': val_loss, 'val_accuracy': val_acc, 'val_auc': val_auc})
pd.DataFrame(results).sort_values('val_auc', ascending=False)

### Actividad 2

Discute si la red con mas parametros fue necesariamente mejor. Relaciona la respuesta con subajuste, sobreajuste y tamano del dataset.

## 5. Evaluacion y umbrales

Una sigmoide entrega una probabilidad estimada. Para convertirla en clase necesitamos un umbral. Cambiar el umbral cambia sensibilidad y especificidad.

In [ ]:
best_name = pd.DataFrame(results).sort_values('val_auc', ascending=False).iloc[0]['modelo']
best_model = trained_models[best_name]
proba_test = best_model.predict(X_test_s, verbose=0).ravel()
y_pred_05 = (proba_test >= 0.5).astype(int)
print('modelo elegido:', best_name)
print('AUC test:', roc_auc_score(y_test, proba_test))
print(confusion_matrix(y_test, y_pred_05))
print(classification_report(y_test, y_pred_05, digits=3))
rows = []
for threshold in [0.3, 0.5, 0.7]:
    y_pred = (proba_test >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    rows.append({'threshold': threshold, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, 'sensibilidad': tp/(tp+fn), 'especificidad': tn/(tn+fp)})
pd.DataFrame(rows)

## 6. Caso nuevo anonimo

Puedes cambiar estos valores para explorar el modelo. No escribas nombres ni identificadores. No uses el resultado como diagnostico.

In [ ]:
persona = pd.DataFrame([{'Pregnancies': 1, 'Glucose': 115, 'BloodPressure': 72, 'SkinThickness': 25, 'Insulin': 90, 'BMI': 28.5, 'DiabetesPedigreeFunction': 0.35, 'Age': 32}])
persona_s = preprocess.transform(persona[feature_names])
probabilidad = best_model.predict(persona_s, verbose=0).ravel()[0]
print(f'Probabilidad estimada por el modelo: {probabilidad:.3f}')
print(f'Porcentaje aproximado: {100 * probabilidad:.1f}%')

## 7. Experimento opcional con GPU

Pima es pequeno y corre bien en CPU. La siguiente celda crea datos sinteticos de alta dimension y una red mucho mas grande. Viene desactivada para evitar esperas largas. En Colab se puede activar con GPU para comparar tiempos.

In [ ]:
RUN_HEAVY_EXPERIMENT = False
if RUN_HEAVY_EXPERIMENT:
    n_samples = 60000
    n_features = 512
    rng = np.random.default_rng(261)
    X_big = rng.normal(size=(n_samples, n_features)).astype('float32')
    w = rng.normal(size=(n_features, 1)).astype('float32')
    logits = X_big @ w + 0.25 * rng.normal(size=(n_samples, 1)).astype('float32')
    y_big = (logits.ravel() > np.median(logits)).astype('float32')
    Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_big, y_big, test_size=0.20, random_state=261, stratify=y_big)
    big_model = keras.Sequential([layers.Input(shape=(n_features,)), layers.Dense(4096, activation='relu'), layers.Dense(4096, activation='relu'), layers.Dense(2048, activation='relu'), layers.Dense(1, activation='sigmoid')])
    big_model.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
    big_model.summary()
    big_model.fit(Xb_train, yb_train, validation_split=0.20, epochs=5, batch_size=512, verbose=1)
    big_model.evaluate(Xb_test, yb_test, verbose=1)

## Cierre

La red neuronal no reemplaza el pensamiento cientifico. El preprocesamiento, la validacion, los umbrales y la comunicacion responsable son parte del modelo.